In [80]:
import pandas as pd

In [81]:
df = pd.read_csv(r'd:\Startup\Project\ai-career-coach\data\processed\jobs_with_skills.csv')

In [82]:
df.isnull().sum()

Job Title                  0
Company Name               0
Location                   0
Experience Required        0
Salary                     0
Skills Required            0
Job Description            0
Posted Date                0
Scrape Date                0
Job Link                   0
Source Platform            0
Standardized_Job_Title     0
days_sinced_posted         1
salary_min                90
salary_max                90
salary_avg                90
Experience_Years           1
combined_text              0
tokens                     0
filtered_tokens            0
lemmatized_tokens          0
processed_text             0
extracted_skills           0
skill_count                0
technical_skills           0
soft_skills                0
business_skills            0
dtype: int64

In [83]:
df = df.dropna(
    subset=[
        "Standardized_Job_Title",
        "extracted_skills"
    ]
)


In [84]:
benchmark_df = df[[
    "Standardized_Job_Title",
        "extracted_skills"
]].copy()



In [85]:
benchmark_df.head()

,Standardized_Job_Title,extracted_skills
0,Data Analyst,"['artificial intelligence', 'research']"
1,Data Analyst,['research']
2,Data Analyst,['research']
3,Data Analyst,"['analysis', 'research', 'data analysis']"
4,Data Analyst,['research']


In [86]:
df.dtypes

Job Title                  object
Company Name               object
Location                   object
Experience Required        object
Salary                     object
Skills Required            object
Job Description            object
Posted Date                object
Scrape Date                object
Job Link                   object
Source Platform            object
Standardized_Job_Title     object
days_sinced_posted        float64
salary_min                float64
salary_max                float64
salary_avg                float64
Experience_Years          float64
combined_text              object
tokens                     object
filtered_tokens            object
lemmatized_tokens          object
processed_text             object
extracted_skills           object
skill_count                 int64
technical_skills           object
soft_skills                object
business_skills            object
dtype: object

In [87]:
type(df['extracted_skills'][0])

str

In [88]:
import ast
benchmark_df['extracted_skills'] = benchmark_df['extracted_skills'].apply(
    ast.literal_eval
)

In [89]:
benchmark_df.head()

,Standardized_Job_Title,extracted_skills
0,Data Analyst,"[artificial intelligence, research]"
1,Data Analyst,[research]
2,Data Analyst,[research]
3,Data Analyst,"[analysis, research, data analysis]"
4,Data Analyst,[research]


In [90]:
type(benchmark_df['extracted_skills'][0])

list

In [91]:
benchmark_df.shape

(165, 2)

In [92]:
benchmark_df = benchmark_df.explode(
    "extracted_skills"
)

In [93]:
benchmark_df.head()

,Standardized_Job_Title,extracted_skills
0,Data Analyst,artificial intelligence
0,Data Analyst,research
1,Data Analyst,research
2,Data Analyst,research
3,Data Analyst,analysis


In [94]:
benchmark_df.shape

(1051, 2)

In [95]:
exclude_skills = [
    "analysis",
    "research",
    "data analysis",
    "communication" , 
    "data visualization" , 
    "leadership" , 
    "presentation"
]

benchmark_df = benchmark_df[~benchmark_df['extracted_skills'].isin(exclude_skills)].copy()

In [96]:
benchmark_df = benchmark_df.rename(
    columns={
        "extracted_skills": "Skill"
    }
)


In [97]:
benchmark_df.shape

(707, 2)

In [98]:
benchmark_df.head()

,Standardized_Job_Title,Skill
0,Data Analyst,artificial intelligence
6,Other,tableau
6,Other,java
6,Other,power bi
6,Other,python


In [99]:
benchmark_df = benchmark_df.reset_index(
    drop=True
)

In [100]:
role_skill_counts = (
    benchmark_df
    .groupby(
        [
            "Standardized_Job_Title",
            "Skill"
        ]
    )
    .size()
    .reset_index(name="Count")
)

In [101]:
role_skill_counts

,Standardized_Job_Title,Skill,Count
0,Analytics,airflow,1
1,Analytics,aws,1
2,Analytics,azure,2
3,Analytics,dashboarding,1
4,Analytics,databricks,1
...,...,...,...
155,Product Analyst,power bi,2
156,Product Analyst,python,5
157,Product Analyst,r,4
158,Product Analyst,sql,4


In [102]:
print(
    role_skill_counts
    .sort_values(
        by="Count",
        ascending=False
    )
    .head(20)
)

    Standardized_Job_Title       Skill  Count
48            Data Analyst         sql     25
76           Data Engineer         sql     21
44            Data Analyst      python     21
60           Data Engineer         etl     21
6                Analytics       excel     19
72           Data Engineer      python     17
12               Analytics    power bi     16
13               Analytics      python     16
43            Data Analyst    power bi     16
53           Data Engineer     airflow     15
103         Data Scientist      python     15
17               Analytics         sql     15
51            Data Analyst     tableau     13
19               Analytics     tableau     12
35            Data Analyst       excel     12
21        Business Analyst       excel     11
75           Data Engineer       spark     11
135                  Other    big data     11
112         Data Scientist  statistics     10
110         Data Scientist         sql     10


In [103]:
role_skill_counts = (
    role_skill_counts.sort_values(
        by = ['Standardized_Job_Title' , 'Count'] ,
        ascending = [True ,False ]
    )
)

In [104]:
role_skill_counts.head()

,Standardized_Job_Title,Skill,Count
6,Analytics,excel,19
12,Analytics,power bi,16
13,Analytics,python,16
17,Analytics,sql,15
19,Analytics,tableau,12


In [105]:
role_skill_counts.reset_index(drop=True)

,Standardized_Job_Title,Skill,Count
0,Analytics,excel,19
1,Analytics,power bi,16
2,Analytics,python,16
3,Analytics,sql,15
4,Analytics,tableau,12
...,...,...,...
155,Product Analyst,tableau,2
156,Product Analyst,big data,1
157,Product Analyst,dashboarding,1
158,Product Analyst,market research,1


In [106]:

print(
    role_skill_counts[
        role_skill_counts[
            "Standardized_Job_Title"
        ] == "Data Analyst"
    ]
    
)

   Standardized_Job_Title                    Skill  Count
48           Data Analyst                      sql     25
44           Data Analyst                   python     21
43           Data Analyst                 power bi     16
51           Data Analyst                  tableau     13
35           Data Analyst                    excel     12
45           Data Analyst                        r     10
50           Data Analyst               statistics      9
34           Data Analyst                      etl      7
42           Data Analyst                   pandas      7
40           Data Analyst                    numpy      6
33           Data Analyst             dashboarding      5
29           Data Analyst                      aws      4
36           Data Analyst                      gcp      4
38           Data Analyst                      kpi      4
30           Data Analyst                    azure      3
39           Data Analyst               matplotlib      3
49           D

In [108]:
TOP_N = 10
top_role_skill = (
    role_skill_counts.groupby(
        'Standardized_Job_Title'
    )
).head(TOP_N)

In [110]:
top_role_skill.shape

(78, 3)

In [111]:
top_role_skill = top_role_skill.reset_index(drop = True)

In [112]:
top_role_skill.head(20)

,Standardized_Job_Title,Skill,Count
0,Analytics,excel,19
1,Analytics,power bi,16
2,Analytics,python,16
3,Analytics,sql,15
4,Analytics,tableau,12
5,Analytics,statistics,3
6,Analytics,azure,2
7,Analytics,etl,2
8,Analytics,pandas,2
9,Analytics,snowflake,2


In [113]:
top_role_skill[
       top_role_skill[
            "Standardized_Job_Title"
        ] == "Data Analyst"
    ]
    

,Standardized_Job_Title,Skill,Count
18,Data Analyst,sql,25
19,Data Analyst,python,21
20,Data Analyst,power bi,16
21,Data Analyst,tableau,13
22,Data Analyst,excel,12
23,Data Analyst,r,10
24,Data Analyst,statistics,9
25,Data Analyst,etl,7
26,Data Analyst,pandas,7
27,Data Analyst,numpy,6


In [115]:
df2 = pd.read_csv(r'd:\Startup\Project\ai-career-coach\data\processed\top_skill_by_role_cleaned.csv')

In [117]:
df2.shape

(80, 3)

In [119]:
df2[
    df2[
            "Standardized_Job_Title"
        ] == "Data Analyst"
    ]

,Standardized_Job_Title,Skill,Count
20,Data Analyst,data visualization,28
21,Data Analyst,sql,25
22,Data Analyst,python,21
23,Data Analyst,power bi,16
24,Data Analyst,tableau,13
25,Data Analyst,excel,12
26,Data Analyst,r,10
27,Data Analyst,statistics,9
28,Data Analyst,etl,7
29,Data Analyst,pandas,7


In [120]:
import logging 
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


TypeError: 'int' object is not callable